# Merge A-vs-B/C/D/E ANOVA Results and Compute Global BH-FDR

This notebook:
1. Loads `A_vs_B.csv`, `A_vs_C.csv`, `A_vs_D.csv`, and `A_vs_E.csv`
2. Cleans accidental `**` formatting in headers/cells
3. Adds a `comparison` column
4. Merges all four result tables
5. Recomputes Benjamini–Hochberg FDR across all raw p-values
6. Adds `q_global_FDR` and `significant_global_FDR`
7. Saves the master CSV
8. Saves a globally significant-only CSV
9. Produces a compact summary by comparison and AOI


In [1]:
# Install once if needed:
# %pip install pandas statsmodels

from pathlib import Path
import pandas as pd
from statsmodels.stats.multitest import multipletests


## 1. Configuration

In [2]:
CSV_FILES = {
    "A_vs_B": "A_vs_B_merged_results_with_FDR.csv",
    "A_vs_C": "A_vs_C_merged_results_with_FDR.csv",
    "A_vs_D": "A_vs_D_merged_results_with_FDR.csv",
    "A_vs_E": "A_vs_E_merged_results_with_FDR.csv",
}

OUTPUT_MASTER = "A_vs_all_master_global_FDR.csv"
OUTPUT_SIGNIFICANT = "A_vs_all_significant_global_FDR.csv"
OUTPUT_SUMMARY = "A_vs_all_global_FDR_summary.csv"

ALPHA = 0.05


## 2. Helper functions

In [3]:
def clean_text(value):
    if pd.isna(value):
        return value
    return str(value).replace("**", "").strip()


def clean_column_name(name):
    return clean_text(name)


def clean_numeric(series):
    return pd.to_numeric(
        series.astype(str)
              .str.replace("**", "", regex=False)
              .str.strip(),
        errors="coerce"
    )


def parse_boolean(series):
    return (
        series.astype(str)
              .str.replace("**", "", regex=False)
              .str.strip()
              .str.lower()
              .isin(["true", "1", "yes", "y"])
    )


## 3. Load and clean each comparison CSV

In [4]:
frames = []

required_columns = [
    "AOI",
    "metric",
    "eta_sq",
    "p",
    "df",
    "n_A",
    "n_C",
    "mean_A",
    "sd_A",
    "mean_C",
    "sd_C",
    "F",
    "levene_p",
    "significant_raw",
    "q_FDR",
    "significant_FDR",
    "eta_band",
]

for comparison, filename in CSV_FILES.items():
    path = Path(filename)

    if not path.exists():
        raise FileNotFoundError(
            f"Could not find {filename}. "
            "Put all four CSVs in the same folder as this notebook "
            "or edit CSV_FILES above."
        )

    df = pd.read_csv(path)

    # Clean headers such as **sd_A or **mean_C
    df.columns = [clean_column_name(c) for c in df.columns]

    missing = [c for c in required_columns if c not in df.columns]

    if missing:
        raise ValueError(
            f"{comparison}: missing columns {missing}\n"
            f"Available columns: {list(df.columns)}"
        )

    # Add comparison label
    df.insert(0, "comparison", comparison)

    # Clean text columns
    for col in ["AOI", "metric", "df", "eta_band"]:
        df[col] = df[col].apply(clean_text)

    # Clean numeric columns
    numeric_cols = [
        "eta_sq",
        "p",
        "n_A",
        "n_C",
        "mean_A",
        "sd_A",
        "mean_C",
        "sd_C",
        "F",
        "levene_p",
        "q_FDR",
    ]

    for col in numeric_cols:
        df[col] = clean_numeric(df[col])

    # Clean booleans
    df["significant_raw"] = parse_boolean(df["significant_raw"])
    df["significant_FDR"] = parse_boolean(df["significant_FDR"])

    frames.append(df)

    print(f"{comparison}: loaded {len(df)} rows from {filename}")


A_vs_B: loaded 24 rows from A_vs_B_merged_results_with_FDR.csv
A_vs_C: loaded 24 rows from A_vs_C_merged_results_with_FDR.csv
A_vs_D: loaded 24 rows from A_vs_D_merged_results_with_FDR.csv
A_vs_E: loaded 24 rows from A_vs_E_merged_results_with_FDR.csv


## 4. Merge the four comparison files

In [5]:
master = pd.concat(frames, ignore_index=True)

print("Total rows:", len(master))
print("Expected rows:", 4 * 3 * 8)

master.head()


Total rows: 96
Expected rows: 96


,comparison,AOI,metric,eta_sq,p,df,n_A,n_C,mean_A,sd_A,mean_C,sd_C,F,levene_p,significant_raw,q_FDR,significant_FDR,eta_band
0,A_vs_B,CTA,VAI (%),0.0588,0.1317,"1, 38",20,20,19.2650,15.6861,26.6800,14.7408,2.3733,0.6110,False,0.287345,False,small
1,A_vs_B,CTA,TTFF (s),0.0401,0.2280,"1, 36",18,20,3.2989,1.0660,2.9015,0.9316,1.5040,0.7223,False,0.342000,False,small
2,A_vs_B,CTA,Time spent (s),0.0135,0.4759,"1, 38",20,20,0.4535,0.2864,0.5120,0.2236,0.5185,0.4107,False,0.601137,False,small
3,A_vs_B,CTA,Fixations,0.0471,0.1783,"1, 38",20,20,15.3000,16.5819,22.9500,18.6420,1.8803,0.7609,False,0.306514,False,small
4,A_vs_B,CTA,Fix. duration (s),0.1459,0.0180,"1, 36",18,20,0.2778,0.0701,0.2320,0.0415,6.1491,0.0945,True,0.086400,False,large


## 5. Validate the master table

In [6]:
expected_rows_per_comparison = 24
errors = []

for comparison in CSV_FILES:
    sub = master[master["comparison"] == comparison]

    if len(sub) != expected_rows_per_comparison:
        errors.append(
            f"{comparison}: expected {expected_rows_per_comparison} rows, found {len(sub)}"
        )

    duplicate_keys = (
        sub.groupby(["AOI", "metric"])
           .size()
    )
    duplicate_keys = duplicate_keys[duplicate_keys > 1]

    if not duplicate_keys.empty:
        errors.append(
            f"{comparison}: duplicate AOI/metric rows found"
        )

if errors:
    raise ValueError(
        "Validation failed:\n" + "\n".join(errors)
    )

print("Validation passed.")
print("Each comparison has 24 rows = 3 AOIs × 8 metrics.")


Validation passed.
Each comparison has 24 rows = 3 AOIs × 8 metrics.


## 6. Compute global BH-FDR across all tests

In [7]:
reject_global, q_global, _, _ = multipletests(
    master["p"].astype(float).values,
    alpha=ALPHA,
    method="fdr_bh"
)

master["q_global_FDR"] = q_global
master["significant_global_FDR"] = reject_global

print("Global BH-FDR computed across", len(master), "tests.")
print(
    "Number significant after global FDR:",
    int(master["significant_global_FDR"].sum())
)


Global BH-FDR computed across 96 tests.
Number significant after global FDR: 23


## 7. Save master CSV

In [8]:
master.to_csv(OUTPUT_MASTER, index=False)
print(f"Saved: {OUTPUT_MASTER}")


Saved: A_vs_all_master_global_FDR.csv


## 8. Save only globally significant results

In [9]:
significant_global = (
    master[master["significant_global_FDR"]]
    .copy()
    .sort_values(
        ["comparison", "AOI", "q_global_FDR", "eta_sq"]
    )
)

significant_global.to_csv(
    OUTPUT_SIGNIFICANT,
    index=False
)

print(f"Saved: {OUTPUT_SIGNIFICANT}")

significant_global[
    [
        "comparison",
        "AOI",
        "metric",
        "eta_sq",
        "p",
        "q_FDR",
        "q_global_FDR",
        "significant_global_FDR",
    ]
]


Saved: A_vs_all_significant_global_FDR.csv


,comparison,AOI,metric,eta_sq,p,q_FDR,q_global_FDR,significant_global_FDR
12,A_vs_B,Headline,Fix. duration (s),0.3015,0.0002,0.002400,0.002743,True
13,A_vs_B,Headline,FFD (s),0.2905,0.0003,0.002400,0.003600,True
20,A_vs_B,Product,Fix. duration (s),0.3844,0.0000,0.000000,0.000000,True
21,A_vs_B,Product,FFD (s),0.2531,0.0009,0.005400,0.008640,True
27,A_vs_C,CTA,Fixations,0.2193,0.0023,0.011600,0.014720,True
31,A_vs_C,CTA,Revisits,0.2101,0.0029,0.011600,0.016000,True
24,A_vs_C,CTA,VAI (%),0.2119,0.0028,0.011600,0.016000,True
36,A_vs_C,Headline,Fix. duration (s),0.3319,0.0001,0.001200,0.002400,True
37,A_vs_C,Headline,FFD (s),0.2790,0.0005,0.004000,0.005333,True
32,A_vs_C,Headline,VAI (%),0.1951,0.0043,0.014743,0.020640,True


## 9. Summary: globally significant effects by comparison and AOI

In [10]:
summary = (
    master
    .groupby(
        ["comparison", "AOI"],
        observed=True
    )["significant_global_FDR"]
    .sum()
    .reset_index(name="n_significant_global_FDR")
)

summary.to_csv(OUTPUT_SUMMARY, index=False)

print(f"Saved: {OUTPUT_SUMMARY}")
summary


Saved: A_vs_all_global_FDR_summary.csv


,comparison,AOI,n_significant_global_FDR
0,A_vs_B,CTA,0
1,A_vs_B,Headline,2
2,A_vs_B,Product,2
3,A_vs_C,CTA,3
4,A_vs_C,Headline,3
5,A_vs_C,Product,2
6,A_vs_D,CTA,2
7,A_vs_D,Headline,3
8,A_vs_D,Product,2
9,A_vs_E,CTA,0


## 10. Pivot summary

In [11]:
summary_pivot = (
    summary
    .pivot(
        index="comparison",
        columns="AOI",
        values="n_significant_global_FDR"
    )
    .fillna(0)
    .astype(int)
)

summary_pivot


AOI,CTA,Headline,Product
comparison,,,
A_vs_B,0,2,2
A_vs_C,3,3,2
A_vs_D,2,3,2
A_vs_E,0,2,2


## 11. Compare old within-comparison FDR vs new global FDR

In [12]:
comparison_check = master[
    [
        "comparison",
        "AOI",
        "metric",
        "p",
        "q_FDR",
        "significant_FDR",
        "q_global_FDR",
        "significant_global_FDR",
        "eta_sq",
    ]
].copy()

comparison_check["changed_after_global_FDR"] = (
    comparison_check["significant_FDR"]
    != comparison_check["significant_global_FDR"]
)

comparison_check.sort_values(
    [
        "changed_after_global_FDR",
        "comparison",
        "AOI",
        "metric",
    ],
    ascending=[False, True, True, True]
)


,comparison,AOI,metric,p,q_FDR,significant_FDR,q_global_FDR,significant_global_FDR,eta_sq,changed_after_global_FDR
28,A_vs_C,CTA,Fix. duration (s),0.0181,0.039491,True,0.064356,False,0.1455,True
35,A_vs_C,Headline,Fixations,0.0161,0.039491,True,0.064356,False,0.1430,True
39,A_vs_C,Headline,Revisits,0.0180,0.039491,True,0.064356,False,0.1386,True
93,A_vs_E,Product,FFD (s),0.0097,0.058200,False,0.040487,True,0.1715,True
5,A_vs_B,CTA,FFD (s),0.0246,0.098400,False,0.084343,False,0.1326,False
...,...,...,...,...,...,...,...,...,...,...
94,A_vs_E,Product,K-coefficient,0.4335,0.690071,False,0.603130,False,0.0171,False
95,A_vs_E,Product,Revisits,0.4888,0.690071,False,0.634119,False,0.0134,False
89,A_vs_E,Product,TTFF (s),0.3296,0.608492,False,0.502552,False,0.0264,False
90,A_vs_E,Product,Time spent (s),0.6197,0.763429,False,0.750120,False,0.0069,False
